# 시드 단어 문맥 감성분석

## 목적

이 노트북은 확장 사전 없이 seed dictionary만 사용한다. DART 사업보고서의 ESG 관련 섹션에서 시드 단어가 포함된 문장을 추출하고, 한국어 금융 감성분석 모델(`snunlp/KR-FinBert-SC`)로 문장 감성을 판정한 뒤, 그 감성을 문장 안에 포함된 시드 단어의 사용 문맥에 귀속한다.

## 핵심 아이디어

단어 자체를 긍정/부정으로 분류하지 않는다. 대신 다음 방식으로 분석한다.

```text
시드 단어 포함 문장 추출 -> 문장 감성분석 -> 감성 결과를 문장 내 시드 단어에 귀속 -> 기업-연도 변수 생성
```

## 생성 변수 예시

- `seed_sentence_count`: 시드 단어가 포함된 ESG 문장 수
- `positive_seed_sentence_count`: 긍정 문맥 ESG 문장 수
- `negative_seed_sentence_count`: 부정 문맥 ESG 문장 수
- `mean_seed_sentence_sentiment`: 시드 문장 평균 감성점수
- `seed_term_context_count`: 시드 단어-문장 매칭 건수
- `positive_seed_term_context_count`: 긍정 문맥에서 사용된 시드 단어 매칭 건수
- `negative_seed_term_context_count`: 부정 문맥에서 사용된 시드 단어 매칭 건수
- `unique_seed_terms_matched`: 기업-연도별 등장한 고유 시드 단어 수

## 연도 매칭

사업보고서 텍스트는 `fiscal_year` 기준이고, ESG 등급은 다음 해 평가값을 사용한다.

```text
esg_year = fiscal_year + 1
```

In [1]:
from pathlib import Path
import html
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")

LOCAL_ROOT = Path.cwd()
ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/My Drive/UD_26"),
    LOCAL_ROOT,
    LOCAL_ROOT.parent,
]


def first_existing(candidates, default=None):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return default if default is not None else candidates[0]


ROOT = first_existing(
    [p for p in ROOT_CANDIDATES if (p / "data").exists() or (p / "final").exists()],
    LOCAL_ROOT,
)
FINAL_DIR = ROOT / "final"
DATA_DIR = ROOT / "data"
DART_DIR = DATA_DIR / "dart"

COMPANY_MASTER_PATH = first_existing([DATA_DIR / "company_master.csv", FINAL_DIR / "company_master.csv"])
SEED_DICTIONARY_PATH = first_existing([DATA_DIR / "seed_dictionary.csv", FINAL_DIR / "seed_dictionary.csv"])
FILING_INDEX_PATH = first_existing([DART_DIR / "filing_index.csv", FINAL_DIR / "filing_index.csv", DATA_DIR / "filing_index.csv"])
RAW_XML_DIR = first_existing([DART_DIR / "raw_xml", FINAL_DIR / "raw_xml", ROOT / "raw_xml"])

GRADE_MAP = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5, "S": 6}
TARGET_SECTIONS = [
    "II. 사업의 내용",
    "IV. 이사의 경영진단 및 분석의견",
    "VI. 이사회 등 회사의 기관에 관한 사항",
]

for label, path in {
    "ROOT": ROOT,
    "COMPANY_MASTER_PATH": COMPANY_MASTER_PATH,
    "SEED_DICTIONARY_PATH": SEED_DICTIONARY_PATH,
    "FILING_INDEX_PATH": FILING_INDEX_PATH,
    "RAW_XML_DIR": RAW_XML_DIR,
}.items():
    print(f"{label}: {path} | {'OK' if path.exists() else 'MISSING'}")

Mounted at /content/drive
ROOT: /content/drive/MyDrive/UD_26 | OK
COMPANY_MASTER_PATH: /content/drive/MyDrive/UD_26/data/company_master.csv | OK
SEED_DICTIONARY_PATH: /content/drive/MyDrive/UD_26/final/seed_dictionary.csv | OK
FILING_INDEX_PATH: /content/drive/MyDrive/UD_26/final/filing_index.csv | OK
RAW_XML_DIR: /content/drive/MyDrive/UD_26/final/raw_xml | OK


In [2]:
company_master = pd.read_csv(COMPANY_MASTER_PATH, dtype={"stock_code": "string"}, encoding="utf-8-sig")
filing_index = pd.read_csv(FILING_INDEX_PATH, dtype={"stock_code": "string", "rcept_no": "string"}, encoding="utf-8-sig")
seed_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")

for df in [company_master, filing_index]:
    df["stock_code"] = df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
    for col in ["fiscal_year", "esg_year"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

print("company_master:", company_master.shape)
print("filing_index:", filing_index.shape)
print("seed_df:", seed_df.shape)
display(seed_df.head(10))

company_master: (381, 13)
filing_index: (381, 13)
seed_df: (30, 6)


,dimension,seed_term,pattern,source_basis,source_titles,notes
0,E,탄소,탄소,Bloomberg ESG climate change; Sautner-style climate exposure,Bloomberg ESG framework; Sautner-style climate change exposure; Alam et al. (2025),carbon / climate exposure core seed
1,E,온실가스,온실가스|GHG,Bloomberg ESG climate change; GHG emissions literature,Bloomberg ESG framework; Sautner-style climate change exposure; Alam et al. (2025),greenhouse gas emissions seed
2,E,탄소중립,탄소중립,Bloomberg ESG climate change; net-zero transition literature,Bloomberg ESG framework; Sautner-style climate change exposure; Alam et al. (2025),net-zero transition Korean term
3,E,넷제로,넷제로|net zero|net-zero,Bloomberg ESG climate change; net-zero transition literature,Bloomberg ESG framework; Sautner-style climate change exposure; Alam et al. (2025),net-zero borrowed term
4,E,재생에너지,재생에너지|renewable energy,Bloomberg ESG water/energy management; climate opportunity literature,Bloomberg ESG framework; Sautner-style climate change exposure; Alam et al. (2025),renewable energy seed
5,E,에너지,에너지,Bloomberg ESG water/energy management,Bloomberg ESG framework; Alam et al. (2025),energy management broad seed
6,E,전력,전력|전력사용량,Bloomberg ESG energy management,Bloomberg ESG framework; Alam et al. (2025),power usage operational seed
7,E,폐기물,폐기물,Bloomberg ESG material and waste,Bloomberg ESG framework; Bao et al. (2024),material and waste seed
8,E,재활용,재활용|자원순환,Bloomberg ESG material and waste,Bloomberg ESG framework; Bao et al. (2024),waste recycling / circularity seed
9,E,폐수,폐수|수질|물관리,Bloomberg ESG water management,Bloomberg ESG framework; Bao et al. (2024),water / wastewater management seed


In [3]:
def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKC", html.unescape(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def strip_tags_keep_space(xml_fragment):
    text = html.unescape(xml_fragment)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return unicodedata.normalize("NFKC", text)


def resolve_xml_path(xml_path_value):
    xml_path_text = str(xml_path_value).replace("\\", "/")
    candidates = [
        ROOT / xml_path_text,
        Path(xml_path_text),
        RAW_XML_DIR / Path(xml_path_text).name,
        FINAL_DIR / "raw_xml" / Path(xml_path_text).name,
        DART_DIR / "raw_xml" / Path(xml_path_text).name,
    ]
    return first_existing(candidates, candidates[0])


title_re = re.compile(r"<TITLE\b[^>]*>(.*?)</TITLE>", flags=re.I | re.S)
main_title_re = re.compile(r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X|Ⅰ|Ⅱ|Ⅲ|Ⅳ|Ⅴ|Ⅵ|Ⅶ|Ⅷ|Ⅸ|Ⅹ)\.")
target_title_regex = {
    "II. 사업의 내용": r"^(II|Ⅱ)\.\s*사업의\s*내용",
    "IV. 이사의 경영진단 및 분석의견": r"^(IV|Ⅳ)\.\s*이사의\s*경영진단\s*및\s*분석의견",
    "VI. 이사회 등 회사의 기관에 관한 사항": r"^(VI|Ⅵ)\.\s*이사회\s*등\s*회사의\s*기관에\s*관한\s*사항",
}


def extract_target_sections(xml_text):
    matches = list(title_re.finditer(xml_text))
    sections = []
    active = None

    for i, match in enumerate(matches):
        title = strip_tags_keep_space(match.group(1))
        next_start = matches[i + 1].start() if i + 1 < len(matches) else len(xml_text)
        body = xml_text[match.end():next_start]

        if main_title_re.search(title):
            active = None
            for section_name, pattern in target_title_regex.items():
                if re.search(pattern, title):
                    active = section_name
                    break
            if active is None:
                continue

        if active:
            section_text = strip_tags_keep_space(body)
            if section_text:
                sections.append({"section": active, "title": title, "text": section_text})

    return sections


def split_sentences(text):
    text = normalize_text(text)
    if not text:
        return []
    # Python re does not allow variable-width look-behind, so split after
    # common sentence endings by capturing the delimiter and rebuilding.
    chunks = re.split(r"([.!?。！？]|다\.|요\.)\s+|\n+", text)
    sentences = []
    buffer = ""
    for chunk in chunks:
        if not chunk:
            continue
        stripped = chunk.strip()
        if re.fullmatch(r"[.!?。！？]|다\.|요\.", stripped):
            buffer += stripped
            sentence = buffer.strip()
            if len(sentence) >= 8:
                sentences.append(sentence)
            buffer = ""
        elif stripped:
            buffer = f"{buffer} {stripped}".strip() if buffer else stripped
    if len(buffer.strip()) >= 8:
        sentences.append(buffer.strip())
    return sentences

In [4]:
section_rows = []
missing_xml_rows = []

for _, row in filing_index.iterrows():
    xml_path = resolve_xml_path(row.get("xml_path", ""))
    if not xml_path.exists():
        missing_xml_rows.append({**row.to_dict(), "resolved_xml_path": str(xml_path)})
        continue
    xml_text = xml_path.read_text(encoding="utf-8", errors="ignore")
    for section in extract_target_sections(xml_text):
        section_rows.append({
            "stock_code": row["stock_code"],
            "company_name": row["company_name"],
            "fiscal_year": int(row["fiscal_year"]),
            "rcept_no": row["rcept_no"],
            "section": section["section"],
            "section_text": normalize_text(section["text"]),
        })

section_df = pd.DataFrame(section_rows)
if section_df.empty:
    raise ValueError("No target sections extracted. Check filing_index xml_path and section extraction rules.")

print("section rows:", section_df.shape)
print("missing xml rows:", len(missing_xml_rows))
display(section_df.groupby("section").size().rename("rows"))

corpus_df = (
    section_df.groupby(["stock_code", "company_name", "fiscal_year", "rcept_no"], as_index=False)
    .agg(document=("section_text", " ".join), section_count=("section", "nunique"))
)
corpus_df["document_norm"] = corpus_df["document"].map(normalize_text)
corpus_df["total_char_count"] = corpus_df["document_norm"].str.len()
corpus_df["total_word_count"] = corpus_df["document_norm"].str.split().str.len()
corpus_df["esg_year"] = corpus_df["fiscal_year"] + 1

print("firm-year corpus rows:", len(corpus_df))
display(corpus_df[["stock_code", "company_name", "fiscal_year", "esg_year", "section_count", "total_word_count"]].head())

section rows: (4155, 6)
missing xml rows: 3


,rows
section,
II. 사업의 내용,2643
IV. 이사의 경영진단 및 분석의견,378
VI. 이사회 등 회사의 기관에 관한 사항,1134


firm-year corpus rows: 378


,stock_code,company_name,fiscal_year,esg_year,section_count,total_word_count
0,000020,동화약품,2022,2023,3,7808
1,000020,동화약품,2023,2024,3,8065
2,000020,동화약품,2024,2025,3,8097
3,000040,KR모터스,2022,2023,3,4201
4,000040,KR모터스,2023,2024,3,4888


In [5]:
def seed_terms_from_row(row):
    values = [row.get("seed_term", "")]
    pattern = row.get("pattern", "")
    if pd.notna(pattern):
        values.extend(str(pattern).split("|"))
    terms = []
    seen = set()
    for value in values:
        term = normalize_text(value).strip()
        if not term or term.lower() == "nan" or term in seen:
            continue
        seen.add(term)
        terms.append(term)
    return terms


seed_records = []
seen = set()
for _, row in seed_df.reset_index(drop=True).iterrows():
    dimension = normalize_text(row.get("dimension", ""))
    if dimension not in {"E", "S", "G"}:
        continue
    seed_term = normalize_text(row.get("seed_term", ""))
    for term in seed_terms_from_row(row):
        key = (dimension, seed_term, term)
        if key in seen:
            continue
        seen.add(key)
        seed_records.append({
            "dimension": dimension,
            "seed_term": seed_term,
            "matched_term": term,
            "regex": re.compile(re.escape(term), flags=re.I),
        })

seed_record_df = pd.DataFrame([{k: v for k, v in rec.items() if k != "regex"} for rec in seed_records])
print("seed records:", len(seed_records))
display(seed_record_df.groupby("dimension").size().rename("terms"))
display(seed_record_df.head(20))

seed records: 54


,terms
dimension,
E,18
G,18
S,18


,dimension,seed_term,matched_term
0,E,탄소,탄소
1,E,온실가스,온실가스
2,E,온실가스,GHG
3,E,탄소중립,탄소중립
4,E,넷제로,넷제로
5,E,넷제로,net zero
6,E,넷제로,net-zero
7,E,재생에너지,재생에너지
8,E,재생에너지,renewable energy
9,E,에너지,에너지


In [6]:
def match_seed_terms(sentence):
    matches = []
    for rec in seed_records:
        count = len(rec["regex"].findall(sentence))
        if count:
            matches.append({
                "dimension": rec["dimension"],
                "seed_term": rec["seed_term"],
                "matched_term": rec["matched_term"],
                "term_occurrences": count,
            })
    return matches


sentence_rows = []
term_context_rows = []

for _, row in corpus_df.iterrows():
    for sent_idx, sentence in enumerate(split_sentences(row["document_norm"])):
        matches = match_seed_terms(sentence)
        if not matches:
            continue

        unique_terms = sorted({m["matched_term"] for m in matches})
        unique_seed_terms = sorted({m["seed_term"] for m in matches})
        dimensions = sorted({m["dimension"] for m in matches})
        sentence_id = f"{row['stock_code']}_{int(row['fiscal_year'])}_{row['rcept_no']}_{sent_idx:05d}"

        base = {
            "sentence_id": sentence_id,
            "stock_code": row["stock_code"],
            "company_name": row["company_name"],
            "fiscal_year": int(row["fiscal_year"]),
            "esg_year": int(row["esg_year"]),
            "rcept_no": row["rcept_no"],
            "total_word_count": row["total_word_count"],
            "total_char_count": row["total_char_count"],
            "section_count": row["section_count"],
            "sentence": sentence,
            "matched_dimensions": "; ".join(dimensions),
            "matched_seed_terms": "; ".join(unique_seed_terms),
            "matched_terms": "; ".join(unique_terms),
            "matched_term_count": len(unique_terms),
            "matched_occurrence_count": int(sum(m["term_occurrences"] for m in matches)),
        }
        sentence_rows.append(base)

        for m in matches:
            term_context_rows.append({
                **{k: base[k] for k in [
                    "sentence_id", "stock_code", "company_name", "fiscal_year", "esg_year", "rcept_no",
                    "total_word_count", "total_char_count", "section_count", "sentence",
                ]},
                "dimension": m["dimension"],
                "seed_term": m["seed_term"],
                "matched_term": m["matched_term"],
                "term_occurrences": int(m["term_occurrences"]),
            })

sentence_df = pd.DataFrame(sentence_rows)
term_context_df = pd.DataFrame(term_context_rows)
if sentence_df.empty:
    raise ValueError("No seed-term sentences were extracted. Check seed dictionary and section extraction.")

print("seed sentence rows:", sentence_df.shape)
print("seed term-context rows:", term_context_df.shape)
display(sentence_df.head(10))
display(term_context_df.groupby("dimension").size().rename("term_context_rows"))

seed sentence rows: (32780, 15)
seed term-context rows: (55636, 14)


,sentence_id,stock_code,company_name,fiscal_year,esg_year,rcept_no,total_word_count,total_char_count,section_count,sentence,matched_dimensions,matched_seed_terms,matched_terms,matched_term_count,matched_occurrence_count
0,000020_2022_20230315001100_00024,000020,동화약품,2022,2023,20230315001100,7808,37953,3,"또한 브랜 드 인지도 및 대중매체 광고 활용, 전문적인 디테일 활동, 영업인력 전문교육을 통 한 영업력 강화 등을 통해 판매를 강화해 나가고 있습니다.",S,교육훈련,교육,1,1
1,000020_2022_20230315001100_00031,000020,동화약품,2022,2023,20230315001100,7808,37953,3,연결기업의 재무부문은 이사회에서 승인된 위험관리 정책 및 절차에 따라 연결기업의 영업과 관련한 금융위험을 감시하고 관리하는 역할을 하고 있습니다.,G,이사회,이사회,1,1
2,000020_2022_20230315001100_00060,000020,동화약품,2022,2023,20230315001100,7808,37953,3,"연결실체의 종속회사의 전환상환우선주 발행에 따른 계약 상황은 아래와 같습니다.1) 메디쎄이 주식회사 발행 전환상환우선주 구 분 내 용 발행자 주식회사 메디쎄이 발행일자 2016-05-31 인수자 IBK기업은행, 아이비케이금융그룹 코넥스투자조합, 글로벌원밸류업전문사모투자신탁3호 주당 액면가(원) 500 주당 발행가(...",G,의결권; 주주,의결권; 주주,2,3
3,000020_2022_20230315001100_00101,000020,동화약품,2022,2023,20230315001100,7808,37953,3,"관계법령 또는 정부의 규제제약산업은 국민의 건강관리 및 질병의 예방, 치료, 처치 등을 위해 의약품을 개발·허가·제조 및 품질관리, 유통·판매하는 산업으로서 타 산업에 비해 많은 규제(안전성·유효성·안정성 확보, 약사법, 약가규제, 지적재산권 등)와 제약이 있습니다.최근 정부는 신약은 패스트트랙을 '지원'하되, 제...",S,안전,안전,1,1
4,000020_2022_20230315001100_00110,000020,동화약품,2022,2023,20230315001100,7808,37953,3,"또한 정부의 약가 적정화 정책 및 한미 FTA체결, 약제비 적정화 방안 시행, 기등재의약품 목록 정비 사업, GMP기준 선진화 추진, 비윤리적 영업관행 금지 등급변하는 경쟁 환경속 에 각 제약사별 실적차별화가 예상되며 제품력, 영업력 및 브랜드인지도 등 경쟁요인과 더불어 신약개발력 및 수출경쟁력 확보 여부가 중요한...",G,윤리,윤리,1,1
5,000020_2022_20230315001100_00115,000020,동화약품,2022,2023,20230315001100,7808,37953,3,"전문의약품시장은 지속적인 대내외 환경변화 요인이 발생하고 있으며, 이러한 환경속에서 준법경영을 위해 CP(공정거래 자율준수프로그램)운영을 강화하는 동시에 제품 및 영업력 강화를 통한 경쟁력 확보에 중점을 두고 있습니다.",G,준법,준법,1,1
6,000020_2022_20230315001100_00133,000020,동화약품,2022,2023,20230315001100,7808,37953,3,"그러나 한편으로는 수요가 급격하게 반등하면서 원자재 가격과 물류비의 상승 등 공급망에 제약을 불러왔고, 기업들에 큰 부담으로 작용하기도 하였습니다.",S,공급망,공급망,1,1
7,000020_2022_20230315001100_00140,000020,동화약품,2022,2023,20230315001100,7808,37953,3,동화약품은 지속 가능한 성장을 위해 ▲ 경쟁력 강화를 통한 지속가능 성장 ▲ 빠르고 유연한 프로세스 정립 ▲ 회사 성장에 기여하는 R&D 구축 ▲ 직원 역량강화를 통한 견고한 조직 구현 ▲ 포트폴리오 확대를 통한 미래 성장동력 확보를 추진하며 모든 역량을 집중하고 있습니다.,S,임직원,직원,1,1
8,000020_2022_20230315001100_00169,000020,동화약품,2022,2023,20230315001100,7808,37953,3,이사회 구성의 개요 당사의 이사회는 2022년 12월 31일 현재 5인의 사내이사와 3인의 사외이사 등 8인의 이사로 구성되어 있습니다.,G,사외이사; 이사회,사외이사; 이사회,2,3
9,000020_2022_20230315001100_00170,000020,동화약품,2022,2023,20230315001100,7808,37953,3,이사회의 의장은 회사의 정관 제 36조 제2항에 의하여 대표이사가 의장이 되고 유고시 이사회에서 정하는 자가 그 직무를 대행합니다.,G,이사회,이사회,1,2


,term_context_rows
dimension,
E,9478
G,36682
S,9476


In [7]:
try:
    import torch
    from transformers import pipeline
except ImportError as exc:
    raise ImportError(
        "감성분석에는 transformers와 torch가 필요합니다. 설치 후 다시 실행하세요: python -m pip install transformers torch"
    ) from exc

SENTIMENT_MODEL = "snunlp/KR-FinBert-SC"
DEVICE = 0 if torch.cuda.is_available() else -1
BATCH_SIZE = 64 if DEVICE == 0 else 32

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model=SENTIMENT_MODEL,
    tokenizer=SENTIMENT_MODEL,
    truncation=True,
    max_length=256,
    device=DEVICE,
)

print("sentiment model:", SENTIMENT_MODEL)
print("device:", "cuda" if DEVICE == 0 else "cpu")
print("batch size:", BATCH_SIZE)
print("label map:", sentiment_pipe.model.config.id2label)

config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/406M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: snunlp/KR-FinBert-SC
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/406M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

sentiment model: snunlp/KR-FinBert-SC
device: cuda
batch size: 64
label map: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [8]:
def parse_sentiment_output(result):
    label_raw = str(result["label"])
    label = label_raw.upper().replace(" ", "_")
    score = float(result["score"])

    if label.startswith("LABEL_"):
        try:
            label_id = int(label.split("_", 1)[1])
            label = str(sentiment_pipe.model.config.id2label.get(label_id, label)).upper().replace(" ", "_")
        except ValueError:
            pass

    if any(token in label for token in ["NEG", "NEGATIVE", "부정", "하락", "악재"]):
        return "negative", -score
    if any(token in label for token in ["POS", "POSITIVE", "긍정", "상승", "호재"]):
        return "positive", score
    if any(token in label for token in ["NEU", "NEUTRAL", "중립"]):
        return "neutral", 0.0
    return label_raw.lower(), np.nan


raw_results = sentiment_pipe(sentence_df["sentence"].fillna("").tolist(), batch_size=BATCH_SIZE)
parsed = [parse_sentiment_output(result) for result in raw_results]

sentence_df["sentiment_label"] = [label for label, _ in parsed]
sentence_df["sentiment_score"] = [score for _, score in parsed]
sentence_df["positive_sentiment"] = (sentence_df["sentiment_label"] == "positive").astype(int)
sentence_df["negative_sentiment"] = (sentence_df["sentiment_label"] == "negative").astype(int)
sentence_df["neutral_sentiment"] = (sentence_df["sentiment_label"] == "neutral").astype(int)

term_context_df = term_context_df.merge(
    sentence_df[["sentence_id", "sentiment_label", "sentiment_score", "positive_sentiment", "negative_sentiment", "neutral_sentiment"]],
    on="sentence_id",
    how="left",
)

print("sentiment label counts")
display(sentence_df["sentiment_label"].value_counts(dropna=False).rename("sentences"))
display(sentence_df[["company_name", "fiscal_year", "matched_terms", "sentiment_label", "sentiment_score", "sentence"]].head(20))

sentiment label counts


,sentences
sentiment_label,
neutral,28248
positive,3483
negative,1049


,company_name,fiscal_year,matched_terms,sentiment_label,sentiment_score,sentence
0,동화약품,2022,교육,neutral,0.000000,"또한 브랜 드 인지도 및 대중매체 광고 활용, 전문적인 디테일 활동, 영업인력 전문교육을 통 한 영업력 강화 등을 통해 판매를 강화해 나가고 있습니다."
1,동화약품,2022,이사회,neutral,0.000000,연결기업의 재무부문은 이사회에서 승인된 위험관리 정책 및 절차에 따라 연결기업의 영업과 관련한 금융위험을 감시하고 관리하는 역할을 하고 있습니다.
2,동화약품,2022,의결권; 주주,negative,-0.697641,"연결실체의 종속회사의 전환상환우선주 발행에 따른 계약 상황은 아래와 같습니다.1) 메디쎄이 주식회사 발행 전환상환우선주 구 분 내 용 발행자 주식회사 메디쎄이 발행일자 2016-05-31 인수자 IBK기업은행, 아이비케이금융그룹 코넥스투자조합, 글로벌원밸류업전문사모투자신탁3호 주당 액면가(원) 500 주당 발행가(..."
3,동화약품,2022,안전,neutral,0.000000,"관계법령 또는 정부의 규제제약산업은 국민의 건강관리 및 질병의 예방, 치료, 처치 등을 위해 의약품을 개발·허가·제조 및 품질관리, 유통·판매하는 산업으로서 타 산업에 비해 많은 규제(안전성·유효성·안정성 확보, 약사법, 약가규제, 지적재산권 등)와 제약이 있습니다.최근 정부는 신약은 패스트트랙을 '지원'하되, 제..."
4,동화약품,2022,윤리,positive,0.978807,"또한 정부의 약가 적정화 정책 및 한미 FTA체결, 약제비 적정화 방안 시행, 기등재의약품 목록 정비 사업, GMP기준 선진화 추진, 비윤리적 영업관행 금지 등급변하는 경쟁 환경속 에 각 제약사별 실적차별화가 예상되며 제품력, 영업력 및 브랜드인지도 등 경쟁요인과 더불어 신약개발력 및 수출경쟁력 확보 여부가 중요한..."
5,동화약품,2022,준법,neutral,0.000000,"전문의약품시장은 지속적인 대내외 환경변화 요인이 발생하고 있으며, 이러한 환경속에서 준법경영을 위해 CP(공정거래 자율준수프로그램)운영을 강화하는 동시에 제품 및 영업력 강화를 통한 경쟁력 확보에 중점을 두고 있습니다."
6,동화약품,2022,공급망,negative,-0.639538,"그러나 한편으로는 수요가 급격하게 반등하면서 원자재 가격과 물류비의 상승 등 공급망에 제약을 불러왔고, 기업들에 큰 부담으로 작용하기도 하였습니다."
7,동화약품,2022,직원,positive,0.999462,동화약품은 지속 가능한 성장을 위해 ▲ 경쟁력 강화를 통한 지속가능 성장 ▲ 빠르고 유연한 프로세스 정립 ▲ 회사 성장에 기여하는 R&D 구축 ▲ 직원 역량강화를 통한 견고한 조직 구현 ▲ 포트폴리오 확대를 통한 미래 성장동력 확보를 추진하며 모든 역량을 집중하고 있습니다.
8,동화약품,2022,사외이사; 이사회,neutral,0.000000,이사회 구성의 개요 당사의 이사회는 2022년 12월 31일 현재 5인의 사내이사와 3인의 사외이사 등 8인의 이사로 구성되어 있습니다.
9,동화약품,2022,이사회,neutral,0.000000,이사회의 의장은 회사의 정관 제 36조 제2항에 의하여 대표이사가 의장이 되고 유고시 이사회에서 정하는 자가 그 직무를 대행합니다.


In [9]:
base_panel = corpus_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year", "rcept_no",
    "total_word_count", "total_char_count", "section_count",
]].copy()

sentence_agg = (
    sentence_df.groupby(["stock_code", "fiscal_year"], as_index=False)
    .agg(
        seed_sentence_count=("sentence_id", "nunique"),
        positive_seed_sentence_count=("positive_sentiment", "sum"),
        negative_seed_sentence_count=("negative_sentiment", "sum"),
        neutral_seed_sentence_count=("neutral_sentiment", "sum"),
        mean_seed_sentence_sentiment=("sentiment_score", "mean"),
        unique_seed_terms_in_sentences=("matched_terms", lambda s: len(set(term.strip() for text in s.dropna() for term in str(text).split(";") if term.strip()))),
    )
)

term_agg = (
    term_context_df.groupby(["stock_code", "fiscal_year"], as_index=False)
    .agg(
        seed_term_context_count=("matched_term", "count"),
        seed_term_occurrence_count=("term_occurrences", "sum"),
        positive_seed_term_context_count=("positive_sentiment", "sum"),
        negative_seed_term_context_count=("negative_sentiment", "sum"),
        neutral_seed_term_context_count=("neutral_sentiment", "sum"),
        mean_seed_term_context_sentiment=("sentiment_score", "mean"),
        unique_seed_terms_matched=("matched_term", "nunique"),
    )
)

feature_df = base_panel.merge(sentence_agg, on=["stock_code", "fiscal_year"], how="left").merge(
    term_agg,
    on=["stock_code", "fiscal_year"],
    how="left",
)

count_cols = [
    "seed_sentence_count", "positive_seed_sentence_count", "negative_seed_sentence_count", "neutral_seed_sentence_count",
    "unique_seed_terms_in_sentences", "seed_term_context_count", "seed_term_occurrence_count",
    "positive_seed_term_context_count", "negative_seed_term_context_count", "neutral_seed_term_context_count",
    "unique_seed_terms_matched",
]
for col in count_cols:
    feature_df[col] = feature_df[col].fillna(0)

for col in ["mean_seed_sentence_sentiment", "mean_seed_term_context_sentiment"]:
    feature_df[col] = feature_df[col].fillna(0)

feature_df["positive_seed_sentence_share"] = feature_df["positive_seed_sentence_count"] / feature_df["seed_sentence_count"].replace(0, np.nan)
feature_df["negative_seed_sentence_share"] = feature_df["negative_seed_sentence_count"] / feature_df["seed_sentence_count"].replace(0, np.nan)
feature_df["neutral_seed_sentence_share"] = feature_df["neutral_seed_sentence_count"] / feature_df["seed_sentence_count"].replace(0, np.nan)
feature_df["positive_seed_term_context_share"] = feature_df["positive_seed_term_context_count"] / feature_df["seed_term_context_count"].replace(0, np.nan)
feature_df["negative_seed_term_context_share"] = feature_df["negative_seed_term_context_count"] / feature_df["seed_term_context_count"].replace(0, np.nan)
feature_df["seed_sentence_per_1000_words"] = 1000 * feature_df["seed_sentence_count"] / feature_df["total_word_count"].replace(0, np.nan)
feature_df["seed_term_context_per_1000_words"] = 1000 * feature_df["seed_term_context_count"] / feature_df["total_word_count"].replace(0, np.nan)
feature_df["seed_term_occurrence_per_1000_words"] = 1000 * feature_df["seed_term_occurrence_count"] / feature_df["total_word_count"].replace(0, np.nan)

term_sentiment_summary = (
    term_context_df.groupby(["dimension", "seed_term", "matched_term"], as_index=False)
    .agg(
        context_rows=("sentence_id", "nunique"),
        total_occurrences=("term_occurrences", "sum"),
        positive_contexts=("positive_sentiment", "sum"),
        negative_contexts=("negative_sentiment", "sum"),
        neutral_contexts=("neutral_sentiment", "sum"),
        mean_context_sentiment=("sentiment_score", "mean"),
    )
)
term_sentiment_summary["positive_context_share"] = term_sentiment_summary["positive_contexts"] / term_sentiment_summary["context_rows"].replace(0, np.nan)
term_sentiment_summary["negative_context_share"] = term_sentiment_summary["negative_contexts"] / term_sentiment_summary["context_rows"].replace(0, np.nan)

print("feature_df:", feature_df.shape)
display(feature_df.head())
display(term_sentiment_summary.sort_values("context_rows", ascending=False).head(30))

feature_df: (378, 29)


,stock_code,company_name,fiscal_year,esg_year,rcept_no,total_word_count,total_char_count,section_count,seed_sentence_count,positive_seed_sentence_count,negative_seed_sentence_count,neutral_seed_sentence_count,mean_seed_sentence_sentiment,unique_seed_terms_in_sentences,seed_term_context_count,seed_term_occurrence_count,positive_seed_term_context_count,negative_seed_term_context_count,neutral_seed_term_context_count,mean_seed_term_context_sentiment,unique_seed_terms_matched,positive_seed_sentence_share,negative_seed_sentence_share,neutral_seed_sentence_share,positive_seed_term_context_share,negative_seed_term_context_share,seed_sentence_per_1000_words,seed_term_context_per_1000_words,seed_term_occurrence_per_1000_words
0,000020,동화약품,2022,2023,20230315001100,7808,37953,3,35,2,2,31,0.018317,14,68,118,2,3,63,-0.000832,14,0.057143,0.057143,0.885714,0.029412,0.044118,4.482582,8.709016,15.112705
1,000020,동화약품,2023,2024,20240319000652,8065,38467,3,35,2,1,32,0.036592,13,67,114,2,2,63,0.008703,13,0.057143,0.028571,0.914286,0.029851,0.029851,4.339740,8.307502,14.135152
2,000020,동화약품,2024,2025,20250318000739,8097,39259,3,37,2,0,35,0.053468,14,68,115,2,0,66,0.029093,14,0.054054,0.000000,0.945946,0.029412,0.000000,4.569594,8.398172,14.202791
3,000040,KR모터스,2022,2023,20230322001182,4201,20136,3,11,1,0,10,0.090888,10,18,38,1,0,17,0.055543,10,0.090909,0.000000,0.909091,0.055556,0.000000,2.618424,4.284694,9.045465
4,000040,KR모터스,2023,2024,20240321002062,4888,22402,3,11,0,1,10,-0.047889,11,16,38,0,1,15,-0.032923,11,0.000000,0.090909,0.909091,0.000000,0.062500,2.250409,3.273322,7.774141


,dimension,seed_term,matched_term,context_rows,total_occurrences,positive_contexts,negative_contexts,neutral_contexts,mean_context_sentiment,positive_context_share,negative_context_share
29,G,주주,주주,10014,17784,552,399,9063,0.015046,0.055123,0.039844
28,G,이사회,이사회,8953,16229,88,148,8717,-0.006888,0.009829,0.016531
23,G,사외이사,사외이사,7687,21270,9,13,7665,-0.000348,0.001171,0.001691
18,G,감사위원회,감사위원회,4648,9367,11,44,4593,-0.005532,0.002367,0.009466
3,E,에너지,에너지,3292,4867,1112,160,2020,0.268045,0.337789,0.048603
43,S,안전,안전,2274,3352,496,83,1695,0.166853,0.218118,0.036500
47,S,임직원,직원,2274,3202,79,140,2055,-0.027239,0.034741,0.061566
36,S,교육훈련,교육,2259,7645,93,85,2081,0.001588,0.041169,0.037627
31,G,준법,준법,1511,4155,11,11,1489,0.000958,0.007280,0.007280
27,G,의결권,의결권,1394,4230,15,10,1369,0.003414,0.010760,0.007174


In [10]:
grade_cols = ["stock_code", "fiscal_year", "industry", "esg_year", "esg_grade", "e_grade", "s_grade", "g_grade"]
grade_df = company_master[grade_cols].copy()
for col in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    grade_df[f"{col}_num"] = grade_df[col].map(GRADE_MAP)

feature_merge = feature_df.copy()
grade_merge = grade_df.copy()
for df in [feature_merge, grade_merge]:
    df["stock_code"] = df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
    for col in ["fiscal_year", "esg_year"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

feature_merge["esg_year"] = feature_merge["fiscal_year"] + 1
analysis_df = feature_merge.merge(grade_merge, on=["stock_code", "fiscal_year", "esg_year"], how="left")

if not (analysis_df["esg_year"].astype("Int64") == analysis_df["fiscal_year"].astype("Int64") + 1).all():
    raise ValueError("Expected esg_year = fiscal_year + 1 after merge.")

print("analysis rows:", len(analysis_df))
print("missing esg_grade_num:", analysis_df["esg_grade_num"].isna().sum())
display(analysis_df[["company_name", "stock_code", "fiscal_year", "esg_year", "esg_grade", "esg_grade_num", "seed_sentence_count", "positive_seed_sentence_count", "negative_seed_sentence_count", "mean_seed_sentence_sentiment"]].head())

analysis rows: 378
missing esg_grade_num: 0


,company_name,stock_code,fiscal_year,esg_year,esg_grade,esg_grade_num,seed_sentence_count,positive_seed_sentence_count,negative_seed_sentence_count,mean_seed_sentence_sentiment
0,동화약품,000020,2022,2023,C,1,35,2,2,0.018317
1,동화약품,000020,2023,2024,C,1,35,2,1,0.036592
2,동화약품,000020,2024,2025,C,1,37,2,0,0.053468
3,KR모터스,000040,2022,2023,D,0,11,1,0,0.090888
4,KR모터스,000040,2023,2024,D,0,11,0,1,-0.047889


In [11]:
from scipy.stats import spearmanr

FEATURES_TO_TEST = [
    "seed_sentence_count",
    "seed_sentence_per_1000_words",
    "positive_seed_sentence_count",
    "negative_seed_sentence_count",
    "neutral_seed_sentence_count",
    "positive_seed_sentence_share",
    "negative_seed_sentence_share",
    "mean_seed_sentence_sentiment",
    "seed_term_context_count",
    "seed_term_occurrence_count",
    "seed_term_context_per_1000_words",
    "seed_term_occurrence_per_1000_words",
    "positive_seed_term_context_count",
    "negative_seed_term_context_count",
    "positive_seed_term_context_share",
    "negative_seed_term_context_share",
    "mean_seed_term_context_sentiment",
    "unique_seed_terms_matched",
    "total_word_count",
]


def spearman_for_feature(data, y_col, x_col):
    tmp = data[[y_col, x_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 3 or tmp[x_col].nunique() < 2 or tmp[y_col].nunique() < 2:
        return len(tmp), np.nan, np.nan
    rho, p_value = spearmanr(tmp[x_col], tmp[y_col])
    return len(tmp), float(rho), float(p_value)


spearman_rows = []
for feature in FEATURES_TO_TEST:
    if feature not in analysis_df.columns:
        continue
    n, rho, p_value = spearman_for_feature(analysis_df, "esg_grade_num", feature)
    spearman_rows.append({"feature": feature, "n": n, "spearman_rho": rho, "p_value": p_value})

spearman_df = pd.DataFrame(spearman_rows).sort_values("spearman_rho", ascending=False).reset_index(drop=True)
display(spearman_df)

,feature,n,spearman_rho,p_value
0,seed_term_occurrence_count,378,0.721106,6.753280e-62
1,seed_term_context_count,378,0.679714,1.464781e-52
2,total_word_count,378,0.653063,2.529959e-47
3,seed_sentence_count,378,0.609911,6.987739e-40
4,unique_seed_terms_matched,378,0.605798,3.125205e-39
5,neutral_seed_sentence_count,378,0.587082,2.179131e-36
6,positive_seed_sentence_count,378,0.559938,1.406746e-32
7,positive_seed_term_context_count,378,0.551829,1.658599e-31
8,positive_seed_sentence_share,378,0.381657,1.486778e-14
9,mean_seed_sentence_sentiment,378,0.373826,5.551813e-14


In [12]:
import statsmodels.api as sm


def zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std


def run_robust_ols(data, y_col, x_cols, min_n=3):
    required_cols = [y_col] + x_cols
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        return None, {"skip_reason": f"missing columns: {missing_cols}", "n": 0}
    reg_df = data[required_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(reg_df) < min_n:
        return None, {"skip_reason": f"too few complete rows after dropna: {len(reg_df)}", "n": int(len(reg_df))}
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(zscore)
    X = sm.add_constant(X, has_constant="add")
    model = sm.OLS(y, X).fit(cov_type="HC3")
    return model, {"skip_reason": "", "n": int(model.nobs)}


MODEL_SPECS = {
    "M0_seed_sentence_count": ["seed_sentence_count"],
    "M1_sentence_sentiment_counts": ["positive_seed_sentence_count", "negative_seed_sentence_count"],
    "M2_sentence_sentiment_counts_with_length": ["positive_seed_sentence_count", "negative_seed_sentence_count", "total_word_count"],
    "M3_mean_sentence_sentiment_with_length": ["mean_seed_sentence_sentiment", "seed_sentence_count", "total_word_count"],
    "M4_term_context_counts": ["positive_seed_term_context_count", "negative_seed_term_context_count", "unique_seed_terms_matched"],
    "M5_term_context_counts_with_length": ["positive_seed_term_context_count", "negative_seed_term_context_count", "unique_seed_terms_matched", "total_word_count"],
    "M6_mean_term_context_sentiment_with_length": ["mean_seed_term_context_sentiment", "seed_term_context_count", "total_word_count"],
}

ols_rows = []
for model_name, x_cols in MODEL_SPECS.items():
    model, info = run_robust_ols(analysis_df, "esg_grade_num", x_cols)
    if model is None:
        ols_rows.append({"model": model_name, "variable": "SKIPPED", **info})
        continue
    for variable in model.params.index:
        ols_rows.append({
            "model": model_name,
            "variable": variable,
            "coef": float(model.params[variable]),
            "std_err": float(model.bse[variable]),
            "p_value": float(model.pvalues[variable]),
            "r2": float(model.rsquared),
            "n": int(model.nobs),
            "skip_reason": "",
        })

ols_df = pd.DataFrame(ols_rows)
display(ols_df)

,model,variable,coef,std_err,p_value,r2,n,skip_reason
0,M0_seed_sentence_count,const,2.619048,0.073550,1.008989e-277,0.226327,378,
1,M0_seed_sentence_count,seed_sentence_count,0.766458,0.116291,4.372120e-11,0.226327,378,
2,M1_sentence_sentiment_counts,const,2.619048,0.077922,1.160465e-247,0.131508,378,
3,M1_sentence_sentiment_counts,positive_seed_sentence_count,0.678335,0.121480,2.351748e-08,0.131508,378,
4,M1_sentence_sentiment_counts,negative_seed_sentence_count,-0.129014,0.121349,2.877071e-01,0.131508,378,
5,M2_sentence_sentiment_counts_with_length,const,2.619048,0.072627,9.097125e-285,0.247181,378,
6,M2_sentence_sentiment_counts_with_length,positive_seed_sentence_count,0.238031,0.126036,5.894600e-02,0.247181,378,
7,M2_sentence_sentiment_counts_with_length,negative_seed_sentence_count,-0.205769,0.097158,3.418419e-02,0.247181,378,
8,M2_sentence_sentiment_counts_with_length,total_word_count,0.743170,0.079334,7.419981e-21,0.247181,378,
9,M3_mean_sentence_sentiment_with_length,const,2.619048,0.070588,2.487284e-301,0.291000,378,


In [13]:
OUTPUT_ANALYSIS_PATH = FINAL_DIR / "v_2_1_seed_term_context_sentiment_analysis.csv"
OUTPUT_SENTENCE_PATH = FINAL_DIR / "v_2_1_seed_term_context_sentences.csv"
OUTPUT_TERM_SUMMARY_PATH = FINAL_DIR / "v_2_1_seed_term_sentiment_summary.csv"

analysis_df.to_csv(OUTPUT_ANALYSIS_PATH, index=False, encoding="utf-8-sig")
sentence_df.to_csv(OUTPUT_SENTENCE_PATH, index=False, encoding="utf-8-sig")
term_sentiment_summary.to_csv(OUTPUT_TERM_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("saved analysis:", OUTPUT_ANALYSIS_PATH, analysis_df.shape)
print("saved sentences:", OUTPUT_SENTENCE_PATH, sentence_df.shape)
print("saved term summary:", OUTPUT_TERM_SUMMARY_PATH, term_sentiment_summary.shape)

saved analysis: /content/drive/MyDrive/UD_26/final/v_2_1_seed_term_context_sentiment_analysis.csv (378, 38)
saved sentences: /content/drive/MyDrive/UD_26/final/v_2_1_seed_term_context_sentences.csv (32780, 20)
saved term summary: /content/drive/MyDrive/UD_26/final/v_2_1_seed_term_sentiment_summary.csv (52, 11)
